# Introduction to Deep Learning, Assignment 2, Task 1

<div style="text-align: right">   </div>

In this notebook we are going to cover two generative models for generating novel images using **PyTorch**:

1. Variational Autoencoders (**VAEs**)
2. Generative Adversarial Networks (**GANs**)

<img src="https://lilianweng.github.io/lil-log/assets/images/three-generative-models.png" width="500">

Your main goal will be to retrain these models on a dataset of your choice and do some experiments on the learned latent space.

When using the Lab Computers, download the Jupyter Notebook to one of the machines first.

If you want to use Google Colab, you should first copy this notebook and enable GPU runtime in 'Runtime -> Change runtime type -> Hardware acceleration -> GPU'.


In [ ]:
# If running in Google Colab, uncomment below to mount your drive:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)


In [ ]:
import os
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Set device (use GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
!wget https://surfdrive.surf.nl/s/wxStiBetTKDJC32/download -O face_dataset_64x64.npy


### Dataset

This dataset is called [Flickr-Faces-HQ Dataset](https://github.com/NVlabs/ffhq-dataset). Here we will use a downsampled version of it (64x64x3) that will speed up all the experiments. [Download](https://surfdrive.surf.nl/s/wxStiBetTKDJC32).

After downloading you should copy it to your working directory (or modify the code to load it from elsewhere).

After running the notebook on this default dataset you then need to find a dataset of your own.


In [ ]:
# Loading real samples and converting to PyTorch tensors (B, C, H, W)
def load_real_samples(scale=False, path='face_dataset_64x64.npy'):
    if not os.path.exists(path):
        # Create a synthetic placeholder dataset if file is not yet downloaded
        print(f"Dataset {path} not found. Using synthetic random dataset for demonstration.")
        X = np.random.randint(0, 256, (100, 64, 64, 3), dtype=np.uint8)
    else:
        X = np.load(path)

    # Convert from uint8 [0, 255] (N, H, W, C) to float32 [0, 1] or [-1, 1] (N, C, H, W)
    X = X.astype(np.float32)
    if scale:
        # Scale to [-1, 1] for GANs (matching Tanh activation output)
        X = (X - 127.5) / 127.5
    else:
        # Scale to [0, 1] for CAE and VAE (matching Sigmoid activation output)
        X = X / 255.0

    # Permute dimensions to PyTorch format: (N, C, H, W)
    X = np.transpose(X, (0, 3, 1, 2))
    return torch.tensor(X, dtype=torch.float32)

# We will use this function to display the output of our models throughout this notebook
def grid_plot(images, epoch='', name='', n=3, save=False, scale=False):
    if isinstance(images, torch.Tensor):
        images = images.detach().cpu().numpy()

    # Transpose back from (N, C, H, W) to (N, H, W, C) for matplotlib
    if images.ndim == 4 and images.shape[1] in [1, 3]:
        images = np.transpose(images, (0, 2, 3, 1))

    if scale:
        # Rescale from [-1, 1] to [0, 1] for plotting
        images = (images + 1.0) / 2.0
    images = np.clip(images, 0.0, 1.0)

    fig, axes = plt.subplots(n, n, figsize=(6, 6))
    for index, ax in enumerate(axes.flat):
        if index < len(images):
            ax.imshow(images[index])
        ax.axis('off')
    fig.suptitle(f"{name} {epoch}", fontsize=14)
    plt.tight_layout()
    if save:
        os.makedirs('results', exist_ok=True)
        filename = f"results/generated_plot_e{epoch:03d}.png"
        plt.savefig(filename)
        plt.close()
    plt.show()

dataset = load_real_samples()
sample_indices = np.random.randint(0, len(dataset), min(9, len(dataset)))
grid_plot(dataset[sample_indices], name='Flickr dataset (64x64x3)', n=3)


## 2.1 Introduction

The generative models that we are going to cover both have the following components:

1. A downsampling architecture (encoder in case of VAE, and discriminator in case of GAN) to either extract features from the data or model its distribution.
2. An upsampling architecture (decoder for VAE, generator for GAN) that will use some kind of latent vector to generate new samples that resemble the data that it was trained on.

Since we are going to be dealing with images, we are going to use convolutional networks for upsampling and downsampling, similar to what you see below.

<img src="https://i2.wp.com/sefiks.com/wp-content/uploads/2018/03/convolutional-autoencoder.png" width="500">

#### Code for building these components in PyTorch:


In [ ]:
class ConvEncoder(nn.Module):
    """
    Build a basic downsampling convolutional network in PyTorch
    """
    def __init__(self, in_channels=3, out_dim=512, filters=128, n_downsampling_layers=4, out_activation='sigmoid'):
        super().__init__()
        layers = []
        c_in = in_channels
        for _ in range(n_downsampling_layers):
            layers.append(nn.Conv2d(c_in, filters, kernel_size=3, stride=2, padding=1))
            layers.append(nn.ReLU(inplace=True))
            c_in = filters
        self.conv_net = nn.Sequential(*layers)
        self.flatten = nn.Flatten()
        # 64x64 after 4 downsamplings (stride 2) becomes 4x4 spatial resolution
        self.fc = nn.Linear(filters * 4 * 4, out_dim)
        self.out_activation = out_activation

    def forward(self, x):
        h = self.conv_net(x)
        h = self.flatten(h)
        out = self.fc(h)
        if self.out_activation == 'sigmoid':
            return torch.sigmoid(out)
        elif self.out_activation == 'tanh':
            return torch.tanh(out)
        return out


class ConvDecoder(nn.Module):
    """
    Build a deconvolutional network for decoding/upscaling latent vectors in PyTorch

    When building the deconvolutional architecture, usually it is best to use the same layer sizes that
    were used in the downsampling network and ConvTranspose2d layers are used instead of Conv2d layers.
    Using matching layers and hyperparameters ensures that the dimensionality of our output matches the
    shape of our input images (64x64x3).
    """
    def __init__(self, latent_dim=512, out_channels=3, filters=128, n_upsampling_layers=4, activation_out='sigmoid'):
        super().__init__()
        self.filters = filters
        self.fc = nn.Linear(latent_dim, filters * 4 * 4)
        layers = []
        for _ in range(n_upsampling_layers):
            layers.append(nn.ConvTranspose2d(filters, filters, kernel_size=3, stride=2, padding=1, output_padding=1))
            layers.append(nn.ReLU(inplace=True))
        self.deconv_net = nn.Sequential(*layers)
        self.conv_out = nn.Conv2d(filters, out_channels, kernel_size=3, stride=1, padding=1)
        self.activation_out = activation_out

    def forward(self, z):
        h = F.relu(self.fc(z))
        h = h.view(-1, self.filters, 4, 4)
        h = self.deconv_net(h)
        out = self.conv_out(h)
        if self.activation_out == 'sigmoid':
            return torch.sigmoid(out)
        elif self.activation_out == 'tanh':
            return torch.tanh(out)
        return out


### Convolutional Autoencoder example

Using these two basic building blocks we can now build a Convolutional Autoencoder (CAE).

<img src="https://lilianweng.github.io/lil-log/assets/images/autoencoder-architecture.png" width="500">

Even though it's not a generative model, CAE is a great way to illustrate how these two components (convolutional and deconvolutional networks) can be used together to reconstruct images.

You can view such model as a compression/dimensionality reduction method as each image gets compressed to a vector of numbers by the encoder and gets decompressed back into an image using the decoder network.


In [ ]:
class CAE(nn.Module):
    def __init__(self, in_channels=3, latent_dim=512, filters=128):
        super().__init__()
        self.encoder = ConvEncoder(in_channels=in_channels, out_dim=latent_dim, filters=filters, out_activation='sigmoid')
        self.decoder = ConvDecoder(latent_dim=latent_dim, out_channels=in_channels, filters=filters, activation_out='sigmoid')

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# Model dimensions and building CAE
latent_dim = 512
num_filters = 128
cae = CAE(in_channels=3, latent_dim=latent_dim, filters=num_filters).to(device)
optimizer = optim.Adam(cae.parameters(), lr=1e-3)
criterion = nn.BCELoss()

# Create DataLoader
dataloader = DataLoader(TensorDataset(dataset), batch_size=64, shuffle=True)

## Training the Convolutional Autoencoder to reconstruct images
epochs = 5 # Increase to 20-50 epochs when training fully
for epoch in range(epochs):
    cae.train()
    total_loss = 0.0
    for (batch_x,) in dataloader:
        batch_x = batch_x.to(device)
        optimizer.zero_grad()
        recon = cae(batch_x)
        loss = criterion(recon, batch_x)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_x)

    avg_loss = total_loss / len(dataset)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

    # Qualitative comparison on sample images
    cae.eval()
    with torch.no_grad():
        samples = dataset[:9].to(device)
        reconstructed = cae(samples)
        grid_plot(samples, epoch+1, name='Original', n=3, save=False)
        grid_plot(reconstructed, epoch+1, name='Reconstructed', n=3, save=False)


*Note: You may experiment with the latent dimensionality and number of filters in your convolutional network to see how it affects the reconstruction quality. Remember that this also affects the size of the model and time it takes to train.*


---
---

## 2.2 Variational Autoencoders (VAEs)

<img src="https://lilianweng.github.io/lil-log/assets/images/vae-gaussian.png" width="500">

#### Encoder network
This defines the approximate posterior distribution $q(z|x)$, which takes as input an observation and outputs parameters for the conditional distribution of the latent representation (mean $\mu$ and log-variance $\log \sigma^2$). We output log-variance instead of variance directly for numerical stability.

#### Decoder network
This defines the conditional distribution $p(x|z)$, taking latent sample $z$ as input and reconstructing the observation. We model the latent prior as a standard unit Gaussian $\mathcal{N}(0, I)$.

#### Reparameterization trick
To sample $z$ differentiably during training, we apply the reparameterization trick:
$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$
where $\sigma = \exp(0.5 \cdot \log \sigma^2)$.

#### Loss Function
The VAE loss consists of two parts:
1. **Reconstruction loss:** Binary cross entropy (or MSE) between reconstructed image and original image.
2. **KL Divergence:** Analytical KL divergence between $\mathcal{N}(\mu, \sigma^2)$ and $\mathcal{N}(0, I)$:
$$\mathcal{L}_{\text{KL}} = -\frac{1}{2} \sum_{j} \left( 1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2 \right)$$


In [ ]:
class VAE(nn.Module):
    def __init__(self, in_channels=3, latent_dim=64, filters=64, n_downsampling_layers=4):
        super().__init__()
        self.latent_dim = latent_dim
        self.filters = filters
        
        # Downsampling convolutional encoder backbone
        layers = []
        c_in = in_channels
        for _ in range(n_downsampling_layers):
            layers.append(nn.Conv2d(c_in, filters, kernel_size=3, stride=2, padding=1))
            layers.append(nn.ReLU(inplace=True))
            c_in = filters
        self.encoder_backbone = nn.Sequential(*layers)
        self.flatten = nn.Flatten()
        
        # Linear projection heads for mean and log-variance
        self.fc_mu = nn.Linear(filters * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(filters * 4 * 4, latent_dim)
        
        # Deconvolutional decoder
        self.decoder = ConvDecoder(latent_dim=latent_dim, out_channels=in_channels, filters=filters, activation_out='sigmoid')

    def encode(self, x):
        h = self.flatten(self.encoder_backbone(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

def vae_loss_function(recon_x, x, mu, logvar):
    # Binary cross entropy reconstruction loss
    bce = F.binary_cross_entropy(recon_x, x, reduction='sum')
    # Analytical KL divergence
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    # Normalize by batch size
    return (bce + kl) / x.size(0)


In [ ]:
# Training the VAE model
latent_dim = 64
vae = VAE(in_channels=3, latent_dim=latent_dim, filters=64).to(device)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)
dataloader = DataLoader(TensorDataset(dataset), batch_size=8, shuffle=True)

epochs = 5 # Set to 20+ epochs for full training
for epoch in range(epochs):
    vae.train()
    total_loss = 0.0
    for (batch_x,) in dataloader:
        batch_x = batch_x.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = vae(batch_x)
        loss = vae_loss_function(recon, batch_x, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_x)

    avg_loss = total_loss / len(dataset)
    print(f"Epoch {epoch+1}/{epochs} - VAE Loss: {avg_loss:.4f}")

    # Generate novel samples from random vectors sampled from the learned latent space
    vae.eval()
    with torch.no_grad():
        latent_vectors = torch.randn(9, latent_dim).to(device)
        generated_images = vae.decoder(latent_vectors)
        grid_plot(generated_images, epoch+1, name='VAE generated images (randomly sampled)', n=3, save=False)


*Note: again, you might experiment with the latent dimensionality, batch size and the architecture of your convolutional nets to see how it affects the generative capabilities of this model.*


---

## 2.3 Generative Adversarial Networks (GANs)

### GAN architecture
Generative adversarial networks consist of two models: a generative model and a discriminative model.

<img src="https://media.springernature.com/original/springer-static/image/chp%3A10.1007%2F978-1-4842-3679-6_8/MediaObjects/463582_1_En_8_Fig1_HTML.jpg" width="500">

**The discriminator** model is a classifier that determines whether a given image looks like a real image from the dataset or like an artificially created image. This is a binary classifier implemented as a convolutional neural network (CNN).

**The generator** model takes random input vectors (noise) and transforms them into images through a deconvolutional neural network.

Over the course of training iterations, the weights in the discriminator and the generator are updated through backpropagation. The discriminator learns to distinguish real images from generated ones, while the generator learns to produce convincing images that deceive the discriminator.


In [ ]:
class Discriminator(nn.Module):
    """
    Discriminator CNN in PyTorch for 64x64 RGB images
    """
    def __init__(self, in_channels=3, filters=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, filters, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(filters, filters, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(filters, filters, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(filters, filters, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(filters * 4 * 4, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


class Generator(nn.Module):
    """
    Generator deconvolutional network with Tanh output activation in [-1, 1]
    """
    def __init__(self, latent_dim=256, out_channels=3, filters=128):
        super().__init__()
        self.decoder = ConvDecoder(latent_dim=latent_dim, out_channels=out_channels, filters=filters, activation_out='tanh')

    def forward(self, z):
        return self.decoder(z)


### Defining custom functions for training your GANs

---


In [ ]:
def train_gan(generator, discriminator, dataset_scaled, latent_dim, n_epochs=20, batch_size=64, lr=0.0002, beta_1=0.5):
    optimizer_d = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta_1, 0.999))
    optimizer_g = optim.Adam(generator.parameters(), lr=lr, betas=(beta_1, 0.999))
    criterion = nn.BCELoss()

    dataloader = DataLoader(TensorDataset(dataset_scaled), batch_size=batch_size, shuffle=True, drop_last=True)

    for epoch in range(n_epochs):
        for (real_images,) in tqdm(dataloader, desc=f"Epoch {epoch+1}/{n_epochs}"):
            current_batch_size = real_images.size(0)
            real_images = real_images.to(device)

            real_labels = torch.ones(current_batch_size, 1, device=device)
            fake_labels = torch.zeros(current_batch_size, 1, device=device)

            # ---------------------
            #  Train Discriminator
            # ---------------------
            optimizer_d.zero_grad()

            # Loss on real images
            outputs_real = discriminator(real_images)
            d_loss_real = criterion(outputs_real, real_labels)

            # Loss on fake images
            z = torch.randn(current_batch_size, latent_dim, device=device)
            fake_images = generator(z)
            outputs_fake = discriminator(fake_images.detach())
            d_loss_fake = criterion(outputs_fake, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            optimizer_d.step()

            # -----------------
            #  Train Generator
            # -----------------
            optimizer_g.zero_grad()
            outputs = discriminator(fake_images)
            g_loss = criterion(outputs, real_labels) # Generator wants discriminator to output 1 (real)
            g_loss.backward()
            optimizer_g.step()

        # Generate and visualize after each epoch
        generator.eval()
        with torch.no_grad():
            noise = torch.randn(9, latent_dim, device=device)
            generated_images = generator(noise)
            grid_plot(generated_images, epoch+1, name='Generated Images', n=3, scale=True)
        generator.train()


In [ ]:
## Build and train the GAN model (scaled to [-1, 1])
latent_dim = 256
generator = Generator(latent_dim=latent_dim, out_channels=3, filters=128).to(device)
discriminator = Discriminator(in_channels=3, filters=128).to(device)

dataset_scaled = load_real_samples(scale=True)
train_gan(generator, discriminator, dataset_scaled, latent_dim=latent_dim, n_epochs=5, batch_size=64)


*Note: the samples generated by small GANs are more diverse when compared to VAEs, however some samples might look strange and do not resemble the data the model was trained on.*


In [ ]:
# Your code: Retrain VAE and GAN on your own dataset, and implement latent space interpolation here
